# 🧹 Modul 02: Data Cleaning & Preprocessing (Jantung Penambangan Data)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/02_Data_Cleaning_and_Preprocessing.ipynb)

> **Repositori Resmi**: [github.com/antonprafanto/data-mining-zero-to-hero](https://github.com/antonprafanto/data-mining-zero-to-hero)  
> **Tingkat Kesulitan**: Pemula Menengah (From Zero to Hero)  
> **Estimasi Waktu Belajar**: 60 – 75 Menit (Santai dan interaktif)  
> **Tujuan Modul**: Menguasai penanganan nilai hilang (MCAR/MAR/MNAR, missingno, SimpleImputer, KNNImputer), deteksi pencilan (IQR & Z-Score, Trimming vs Capping), penanganan data duplikat, penskalaan fitur (MinMax, Standard, Robust Scaler), serta pengkodean kategori (Ordinal vs One-Hot Encoding).

---

## 💡 1. Kenapa Preprocessing Adalah Jantung Data Mining?

> *"80% waktu seorang praktisi data dihabiskan untuk membersihkan dan menyiapkan data, sedangkan melatih modelnya sendiri hanya butuh 20% waktu."*

Ada pepatah legendaris dalam dunia ilmu data:
### 🗑️ **"Garbage In, Garbage Out" (Sampah Masuk, Sampah Pula yang Keluar)**
Jika Anda menyuapi algoritma tercanggih di dunia dengan data yang kotor, bolong-bolong, salah ketik, dan penuh angka anomali, maka hasil prediksi atau rekomendasinya pasti akan **menyesatkan dan membahayakan keputusan bisnis**.

Mari kita bedah seluruh teknik pembersihan data secara bertahap dan mudah dipahami!

---

## 📦 Menyiapkan Dataset Studi Kasus: Rekam Medis & Nasabah Kotor

Mari kita buat dataset realistis yang mengandung semua jenis kotoran data nyata: data bolong (`NaN`), outlier ekstrem, baris ganda (duplikat), dan salah tipe data teks:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset simulasi pengajuan pinjaman nasabah (mengandung missing value, outlier, duplikat)
data_mentah = {
    'ID_Nasabah': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10], # Baris ke-10 dan 11 duplikat!
    'Usia': [25, np.nan, 28, 45, 120, 32, 29, np.nan, 50, 36, 36],             # NaN & Outlier 120 thn
    'Gaji_Juta': [5.0, 7.5, np.nan, 15.0, 12.0, 8.0, 6.5, 4.5, 95.0, 11.0, 11.0],  # NaN & Outlier 95 Juta
    'Tingkat_Pendidikan': ['S1', 'D3', 'SMA', 'S2', 'S1', 'S1', 'D3', 'SMA', 'S2', np.nan, np.nan],
    'Status_Pernikahan': ['Lajang', 'Menikah', 'Lajang', np.nan, 'Menikah', 'Lajang', 'Menikah', 'Lajang', 'Menikah', 'Menikah', 'Menikah'],
    'Skor_Kredit': [650, 710, 590, 780, 680, 620, np.nan, 580, 800, 720, 720]
}

df = pd.DataFrame(data_mentah)
print("=== DATASET MENTAH (KOTOR & BELUM SIAP) ===")
df

---

## 👯 2. Langkah Pertama: Membersihkan Baris Duplikat (*Duplicate Rows*)

Data ganda sering terjadi karena pelanggan menekan tombol kirim formulir dua kali atau kegagalan sistem database. Data duplikat harus dibuang terlebih dahulu agar tidak merusak perhitungan statistik:

In [ ]:
# 1. Memeriksa jumlah baris duplikat
jumlah_duplikat = df.duplicated().sum()
print(f"Ditemukan {jumlah_duplikat} baris data duplikat persis!")

# 2. Menghapus baris duplikat dengan .drop_duplicates()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Data setelah duplikat dibuang: {df.shape[0]} baris tersisa. ✅")

---

## 🩹 3. Penanganan Nilai Hilang (*Missing Values*)

### 📖 3 Jenis Missing Values Menurut Literatur Akademik:
1. **MCAR (*Missing Completely at Random*)**: Data hilang murni karena kebetulan acak (misal: formulir tertiup angin tanpa pola).
2. **MAR (*Missing at Random*)**: Data hilang berkaitan dengan variabel lain yang tercatat (misal: usia muda lebih jarang mengisi kolom gaji).
3. **MNAR (*Missing Not at Random*)**: Data hilang justru karena nilai dari variabel itu sendiri (misal: orang super kaya sengaja mengosongkan kolom penghasilan karena privasi).

### 📊 Visualisasi Data Kosong dengan Pustaka Khusus: `missingno`

In [ ]:
import missingno as msno

# 1. Diagram Matriks missingno: Garis putih menandakan data yang bolong/hilang
plt.figure(figsize=(8, 3.5))
msno.matrix(df, figsize=(8, 3), sparkline=False)
plt.title('Visualisasi Pola Nilai Kosong dengan missingno (Putih = NaN)', fontsize=12, fontweight='bold')
plt.show()

# 2. Diagram Batang kelengkapan data
msno.bar(df, figsize=(8, 3), color='teal', fontsize=10)
plt.show()

### 🛠️ Strategi Imputasi: SimpleImputer (Mean/Median/Modus) vs KNNImputer

1. **Median (Nilai Tengah)**: Paling direkomendasikan untuk data angka miring (*skewed*) atau yang mengandung outlier.
2. **Modus (*Most Frequent*)**: Paling direkomendasikan untuk data teks kategori.
3. **KNNImputer**: Imputasi cerdas berbasis machine learning! Komputer mencari **$K$ baris nasabah yang paling mirip karakteristiknya**, lalu mengambil rata-rata nilai nasabah terdekat tersebut untuk mengisi data yang bolong!

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

# A. Imputasi Dasar: SimpleImputer
# Mengisi data numerik dengan Median dan data teks dengan Modus (most_frequent)
imputer_num = SimpleImputer(strategy='median')
df[['Usia', 'Gaji_Juta', 'Skor_Kredit']] = imputer_num.fit_transform(df[['Usia', 'Gaji_Juta', 'Skor_Kredit']])

imputer_cat = SimpleImputer(strategy='most_frequent')
df[['Tingkat_Pendidikan', 'Status_Pernikahan']] = imputer_cat.fit_transform(df[['Tingkat_Pendidikan', 'Status_Pernikahan']])

print("Status Missing Values Setelah Imputasi:")
print(df.isna().sum())  # Semua sudah 0 (bebas dari data bolong)! ✅

In [ ]:
# B. Pengenalan Imputasi Cerdas: KNNImputer (Alternatif Canggih)
# Contoh: Menebak angka yang bolong berdasarkan 2 tetangga terdekat (K=2)
data_uji_knn = pd.DataFrame({
    'Usia': [25, 26, 45, np.nan],
    'Gaji_Juta': [5.0, 5.2, 15.0, 5.1]  # Baris terakhir gajinya mirip baris 1 & 2
})
knn_imp = KNNImputer(n_neighbors=2)
hasil_knn = knn_imp.fit_transform(data_uji_knn)
print("Hasil Imputasi Cerdas KNNImputer (Usia baris ke-4 ditebak 25.5 thn karena gajinya mirip!):")
print(pd.DataFrame(hasil_knn, columns=['Usia', 'Gaji_Juta']))

---

## 🕵️ 4. Deteksi & Penanganan Pencilan (*Outliers*)

### Apa Itu Outlier?
Outlier adalah data yang nilainya **menyimpang terlalu jauh dari mayoritas data lainnya**.
- Pada data kita:
  - `Usia`: Ada nasabah berusia **120 tahun** (salah ketik saat pendaftaran).
  - `Gaji_Juta`: Ada nasabah berpenghasilan **Rp 95 Juta**, sementara yang lain di kisaran 5–15 Juta.

### A. Deteksi Visual: Boxplot
Diagram Boxplot secara otomatis menandai outlier sebagai titik-titik kecil di luar garis kumis (*whiskers*):

In [ ]:
plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
sns.boxplot(x=df['Usia'], color='lightcoral')
plt.title('Deteksi Outlier Usia (Titik di 120 thn!)', fontweight='bold')

plt.subplot(1, 2, 2)
sns.boxplot(x=df['Gaji_Juta'], color='skyblue')
plt.title('Deteksi Outlier Gaji (Titik di 95 Juta!)', fontweight='bold')
plt.tight_layout()
plt.show()

### B. Deteksi Matematis 1: Metode IQR (*Interquartile Range*)
Metode standar dunia berbasis kuartil yang sangat tahan terhadap outlier:
1. Hitung Kuartil 1 ($Q1$ = 25%) dan Kuartil 3 ($Q3$ = 75%).
2. Hitung rentang $IQR = Q3 - Q1$.
3. **Batas Bawah** = $Q1 - (1.5 \times IQR)$
4. **Batas Atas** = $Q3 + (1.5 \times IQR)$

Semua angka di luar batas ini resmi dianggap sebagai **Outlier**!

In [ ]:
# Menghitung Batas IQR untuk Kolom Usia
Q1_usia = df['Usia'].quantile(0.25)
Q3_usia = df['Usia'].quantile(0.75)
IQR_usia = Q3_usia - Q1_usia

batas_bawah_usia = Q1_usia - (1.5 * IQR_usia)
batas_atas_usia = Q3_usia + (1.5 * IQR_usia)

print(f"Q1 Usia: {Q1_usia:.1f}, Q3 Usia: {Q3_usia:.1f}, IQR: {IQR_usia:.1f}")
print(f"Batas Wajar Usia: {batas_bawah_usia:.1f} tahun s/d {batas_atas_usia:.1f} tahun")

### C. Deteksi Matematis 2: Metode Z-Score
Metode Z-Score mengukur berapa standar deviasi suatu titik data menyimpang dari rata-rata:
$$Z = \frac{X - \mu}{\sigma}$$
- Aturan Standar: Data dengan nilai $|Z| > 3$ (atau $|Z| > 2.5$ pada data kecil) dikategorikan sebagai outlier ekstrem.

In [ ]:
from scipy import stats

# Menghitung Z-Score untuk kolom Gaji
z_gaji = np.abs(stats.zscore(df['Gaji_Juta']))
outliers_z = df[z_gaji > 2.0] # Threshold Z > 2.0
print(f"Outlier Gaji yang Terdeteksi Menggunakan Z-Score:")
print(outliers_z[['ID_Nasabah', 'Gaji_Juta']])

### D. Dua Teknik Mengatasi Outlier: Trimming vs Capping (Winsorizing)
1. **Trimming (Hapus Baris)**: Membuang baris data yang mengandung outlier. Hanya dipakai jika baris tersebut salah input nyata dan data berlimpah.
2. **Capping / Winsorizing (Membatasi Nilai)**: **Paling Direkomendasikan!** Nilai yang melebihi batas atas dipangkas (*clipped*) ke batas wajar maksimum agar informasi nasabah tidak hilang.

In [ ]:
# Capping nilai Usia dan Gaji menggunakan fungsi .clip()
df['Usia_Bersih'] = df['Usia'].clip(lower=batas_bawah_usia, upper=batas_atas_usia)
df['Gaji_Bersih'] = df['Gaji_Juta'].clip(lower=df['Gaji_Juta'].quantile(0.05), upper=df['Gaji_Juta'].quantile(0.90))

print("Data Sebelum vs Sesudah Penskorsan Outlier (Capping):")
print(df[['ID_Nasabah', 'Usia', 'Usia_Bersih', 'Gaji_Juta', 'Gaji_Bersih']].head(6))

---

## ⚖️ 5. Penskalaan Fitur Numerik (*Feature Scaling*)

### Mengapa Scaling Itu Wajib?
Perhatikan dua kolom ini:
- Kolom `Usia`: Rentang angkanya hanya puluhan (**20 – 50**).
- Kolom `Skor_Kredit`: Rentang angkanya ratusan (**500 – 800**).
- Kolom `Gaji`: Rentang angkanya jutaan (**5.000.000 – 15.000.000**).

Algoritma data mining yang berbasis jarak (seperti **K-NN, K-Means Clustering, SVM**) akan mengira bahwa kolom Gaji jauh lebih penting daripada Usia hanya karena nominal angkanya lebih besar!  
👉 **Solusi:** Kita harus menyamakan skala semua kolom agar adil!

### 3 Metode Penskalaan Terpopuler:
| Scaler | Rumus / Cara Kerja | Rentang Hasil | Kapan Dipakai? |
| :--- | :--- | :--- | :--- |
| **MinMaxScaler** | $\frac{X - X_{min}}{X_{max} - X_{min}}$ | Tepat **0.0 sampai 1.0** | Bagus jika data tidak memiliki outlier ekstrem dan algoritma butuh batas pasti (seperti neural network). |
| **StandardScaler** | $\frac{X - \mu}{\sigma}$ (Z-Score) | Rata-rata 0, Std 1 | **Paling Populer & Standar!** Mengubah data menjadi distribusi normal baku. |
| **RobustScaler** | $\frac{X - Median}{IQR}$ | Berbasis kuartil | **Pilihan Terbaik jika data masih mengandung outlier!** |

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

fitur_angka = df[['Usia_Bersih', 'Gaji_Bersih', 'Skor_Kredit']]

# 1. MinMaxScaler (Rentang 0 s/d 1)
mms = MinMaxScaler()
data_minmax = mms.fit_transform(fitur_angka)

# 2. StandardScaler (Rata-rata 0, Std 1)
std = StandardScaler()
data_std = std.fit_transform(fitur_angka)

# 3. RobustScaler (Tahan Outlier berbasis Median & IQR)
rob = RobustScaler()
data_robust = rob.fit_transform(fitur_angka)

print("=== CONTOH HASIL STANDARD SCALER (Rata-rata 0, Std 1) ===")
print(pd.DataFrame(data_std, columns=['Usia_STD', 'Gaji_STD', 'Skor_STD']).head(3))

print("\n=== CONTOH HASIL MINMAX SCALER (0 s/d 1) ===")
print(pd.DataFrame(data_minmax, columns=['Usia_MMS', 'Gaji_MMS', 'Skor_MMS']).head(3))

---

## 🏷️ 6. Pengkodean Data Kategorikal (*Categorical Encoding*)

Algoritma komputer hanya bisa memproses angka matematika, bukan teks string. Maka kita harus mengubah teks kategori menjadi angka.

Gunakan aturan pemilihan ini:
1. **Ordinal Encoding**: Gunakan jika kategori **memiliki tingkatan/peringkat yang jelas**  
   *Contoh:* Pendidikan: `SMA = 0`, `D3 = 1`, `S1 = 2`, `S2 = 3`.
2. **One-Hot Encoding**: Gunakan jika kategori **tidak memiliki tingkatan (nominal / setara)**  
   *Contoh:* Status Pernikahan: dibuat kolom biner baru `Menikah` (1 jika menikah, 0 jika lajang).  
   *Kiat:* Selalu gunakan parameter `drop_first=True` untuk menghindari jebakan redundansi (*Dummy Variable Trap*).

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# 1. Ordinal Encoding untuk Tingkat Pendidikan (Ada Peringkat!)
hierarki_pendidikan = [['SMA', 'D3', 'S1', 'S2']]
oe = OrdinalEncoder(categories=hierarki_pendidikan)
df['Pendidikan_Encoded'] = oe.fit_transform(df[['Tingkat_Pendidikan']])

# 2. One-Hot Encoding untuk Status Pernikahan (Nominal / Tanpa Peringkat)
# Menggunakan pd.get_dummies dengan drop_first=True
df_final = pd.get_dummies(df, columns=['Status_Pernikahan'], drop_first=True, dtype=int)

print("=== DATASET AKHIR YANG BERSIH, TERSTANDARISASI & SIAP DILATIH! ===")
kolom_siap = ['ID_Nasabah', 'Usia_Bersih', 'Gaji_Bersih', 'Skor_Kredit', 'Pendidikan_Encoded', 'Status_Pernikahan_Menikah']
df_final[kolom_siap]

---

## 🧠 7. Kuis Evaluasi Pemahaman Mandiri

Uji pemahaman Anda terhadap seluruh pilar data cleaning & preprocessing:

In [ ]:
# Evaluasi Pemahaman Mandiri Modul 2
evaluasi_m2 = [
    {
        'no': 1,
        'soal': 'Apa perbedaan mendasar MCAR (Missing Completely at Random) dan MNAR (Missing Not at Random)?',
        'jawaban': 'MCAR terjadi murni karena kebetulan acak tanpa pola, sedangkan MNAR terjadi karena nilai dari variabel itu sendiri (misal orang kaya merahasiakan gajinya).'
    },
    {
        'no': 2,
        'soal': 'Bagaimana prinsip kerja KNNImputer dalam mengisi data yang hilang?',
        'jawaban': 'Mencari K baris tetangga yang paling mirip karakteristiknya, lalu mengambil rata-rata nilai tetangga tersebut untuk mengisi data yang bolong.'
    },
    {
        'no': 3,
        'soal': 'Apa rumus batas atas outlier menggunakan metode IQR?',
        'jawaban': 'Batas Atas = Q3 + (1.5 × IQR). Data di atas angka ini dianggap outlier.'
    },
    {
        'no': 4,
        'soal': 'Kapan sebaiknya kita memilih RobustScaler dibanding StandardScaler?',
        'jawaban': 'Ketika data masih mengandung nilai pencilan (outlier) ekstrem yang tidak ingin kita buang, karena RobustScaler menggunakan Median dan IQR.'
    },
    {
        'no': 5,
        'soal': 'Mengapa kita perlu menambahkan parameter drop_first=True saat melakukan One-Hot Encoding?',
        'jawaban': 'Untuk menghindari jebakan Dummy Variable Trap (multikolinearitas sempurna antar kolom biner).'
    }
]

for ev in evaluasi_m2:
    print(f"Soal {ev['no']}: {ev['soal']}")
    print(f"👉 Kunci: {ev['jawaban']}\n")
print("Jika Anda memahami ke-5 pilar di atas, Anda telah menguasai seni Data Preprocessing! 🌟")

---

## ✍️ 8. Kotak Latihan Bebas (*Playground Challenge*)

Selesaikan tantangan terpandu di bawah ini untuk menguji keahlian baru Anda:

In [ ]:
# TANTANGAN MANDIRI:
# Diberikan data tinggi badan pasien rumah sakit berikut:
tinggi_pasien = pd.Series([160, 165, np.nan, 170, 155, 290, 168]) # Ada 1 NaN dan 1 Outlier ekstrem (290 cm!)

# 1. Isi missing value dengan nilai Median
tinggi_bersih = tinggi_pasien.fillna(tinggi_pasien.median())

# 2. Batasi nilai maksimal wajar di angka 190 cm menggunakan .clip()
tinggi_final = tinggi_bersih.clip(upper=190)

print("Data Tinggi Pasien Sebelum dan Sesudah Dibersihkan:")
print(pd.DataFrame({'Tinggi_Awal': tinggi_pasien, 'Tinggi_Bersih_Final': tinggi_final}))

---

## 🎯 9. Checklist Kelulusan Modul 02 (*Hero Checkpoint*)

Beri tanda centang pada hati Anda jika Anda sudah menguasai poin-poin berikut:
- [ ] Saya paham pepatah *Garbage In, Garbage Out*.
- [ ] Saya bisa membuang data duplikat dengan `.drop_duplicates()`.
- [ ] Saya tahu perbedaan konsep data hilang: MCAR, MAR, dan MNAR.
- [ ] Saya bisa memvisualisasikan data kosong menggunakan pustaka `missingno`.
- [ ] Saya bisa mengisi missing values dengan SimpleImputer (Median & Modus) serta KNNImputer.
- [ ] Saya bisa mendeteksi outlier menggunakan grafik Boxplot, rumus IQR, dan Z-Score.
- [ ] Saya tahu perbedaan Trimming (hapus baris) vs Capping (pangkas batas).
- [ ] Saya paham mengapa fitur numerik wajib disetarakan skalanya (*Feature Scaling*).
- [ ] Saya bisa memilih antara MinMaxScaler, StandardScaler, atau RobustScaler.
- [ ] Saya bisa membedakan Ordinal Encoding (bertingkat) dan One-Hot Encoding (nominal setara).

---

## 🏁 Gerbang Menuju Modul 03: EDA & Feature Engineering!

Kerja yang luar biasa! Data Anda kini sudah bersih, rapi, dan memiliki format angka yang adil untuk diproses oleh algoritma machine learning.

Langkah berikutnya adalah **mengekstrak wawasan tersembunyi dan menciptakan variabel-variabel baru yang lebih cerdas**:

👉 **[Buka Modul 03: Exploratory Data Analysis & Feature Engineering](03_EDA_and_Feature_Engineering.ipynb)**

Sampai jumpa di Modul 03! Tetap semangat dan jadilah Hero Data Mining! 🚀